In [1]:
!pip install mlflow
!pip install neuralforecast
!pip install dagshub
import mlflow
import dagshub
import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATS
import warnings
warnings.filterwarnings('ignore')

REPO_OWNER = "ejoba22"  
REPO_NAME = "walmart-sales-forecasting"

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

mlflow.set_tracking_uri(f"https://dagshub.com/{REPO_OWNER}/{REPO_NAME}.mlflow")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 86.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 99.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 73.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.6/954.6 kB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.9/214.9 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=70d66c08-cd50-45d2-9d7d-e47c24968741&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=762478887760068149aae26b56da80db240b5d13053256df9bde7750064371d1




Accessing as tvada22

Initialized MLflow to track repo "ejoba22/walmart-sales-forecasting"

Repository ejoba22/walmart-sales-forecasting initialized!

In [2]:
# 2. DATA LOADING & TENSOR FORMATTING
DATA_BASE_PATH = '/kaggle/input/notebooks/elenejobava/walmart-eda-feature-engineering/'
train_fe = pd.read_parquet(DATA_BASE_PATH + 'train_features.parquet')

# NeuralForecast STRICTLY requires 3 specific column names to build its tensors:
# 'unique_id' (The specific time-series), 'ds' (The datetime), and 'y' (The target)
df_dl = train_fe.copy()
df_dl = df_dl.rename(columns={
    'Store_Dept': 'unique_id',
    'Date': 'ds',
    'Weekly_Sales': 'y'
})

# Calculate weights for custom Kaggle WMAE
df_dl['weight'] = np.where(df_dl['IsHoliday'] == 1, 5, 1)

# Sort strictly by time so sequences are fed in order
df_dl = df_dl.sort_values(['unique_id', 'ds']).reset_index(drop=True)

In [3]:
# 3. TIME-SERIES SPLIT
# We use the same 12-week cutoff to keep comparisons fair
split_date = df_dl['ds'].max() - pd.Timedelta(weeks=12)

train_df = df_dl[df_dl['ds'] <= split_date]
val_df = df_dl[df_dl['ds'] > split_date]

In [4]:
# 4. CUSTOM METRICS
def calculate_wape(y_true, y_pred):
    return (np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))) * 100

def calculate_wmae(y_true, y_pred, weights):
    return np.sum(weights * np.abs(y_true - y_pred)) / np.sum(weights)

In [5]:
# 5. MODEL TRAINING & TRACKING

# 1. Safely One-Hot Encode categorical columns WITHOUT destroying 'unique_id' or 'ds'
cat_cols = train_df.select_dtypes(include=['object', 'category']).columns.tolist()
cat_cols = [col for col in cat_cols if col not in ['unique_id', 'ds']]

train_df = pd.get_dummies(train_df, columns=cat_cols, dtype='float32')
val_df = pd.get_dummies(val_df, columns=cat_cols, dtype='float32') 

# 2. Drop rows with NaN values
train_df = train_df.dropna()
val_df = val_df.dropna()

# 3. NEW: Filter out any time series that are too short for NBEATS
# We must have at least 'input_size' (52) rows per unique_id to train
min_required_rows = 52 
series_lengths = train_df.groupby('unique_id').size()
valid_uids = series_lengths[series_lengths >= min_required_rows].index

# Keep only the valid series
train_df = train_df[train_df['unique_id'].isin(valid_uids)]

# Ensure the validation set only predicts for valid UIDs we actually trained on
val_df = val_df[val_df['unique_id'].isin(valid_uids)]


# Define the model architecture
model = NBEATS(
    h=12,                 
    input_size=52,        
    max_steps=500,        
    learning_rate=1e-3,
    scaler_type='robust', 
    random_seed=42
)

# Wrap the model in the Nixtla core object
nf = NeuralForecast(
    models=[model],
    freq='W-FRI'              # 'W' tells the PyTorch dataloader our time steps are Weekly
)

mlflow.set_experiment("NBEATS_Training")

with mlflow.start_run(run_name="NBEATS_Baseline"):
    
    # 1. Fit the PyTorch model (now using the cleaned dataframe)
    nf.fit(df=train_df)
    
    # 2. Predict the validation window
    # NeuralForecast automatically takes the last 52 weeks of train_df to predict the next 12 weeks
    val_preds_df = nf.predict().reset_index()

    val_df['ds'] = pd.to_datetime(val_df['ds'])
    val_preds_df['ds'] = pd.to_datetime(val_preds_df['ds'])
    
    # Merge predictions back with actuals to calculate metrics
    results = val_df[['unique_id', 'ds', 'y', 'weight']].merge(val_preds_df, on=['unique_id', 'ds'], how='inner')

    print(f"Number of rows in results after merge: {len(results)}")
    
    wmae_score = calculate_wmae(results['y'], results['NBEATS'], results['weight'])
    wape_score = calculate_wape(results['y'], results['NBEATS'])
    
    # Log manual metrics (Nixtla doesn't support autologging yet)
    mlflow.log_param("horizon", 12)
    mlflow.log_param("input_size", 52)
    mlflow.log_param("max_steps", 500)
    mlflow.log_metric("validation_WMAE", wmae_score)
    mlflow.log_metric("validation_WAPE_percent", wape_score)
    
    print(f"Validation WMAE: {wmae_score:.2f}")
    print(f"Human-Readable Error (WAPE): {wape_score:.2f}%")

Seed set to 42
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
Initializing distributed: GLOBAL_RANK: 0, MEMBER: 1/2
Initializing distributed: GLOBAL_RANK: 1, MEMBER: 2/2
----------------------------------------------------------------------------------------------------
distributed_backend=nccl
All distributed processes registered. Starting with 2 processes
----------------------------------------------------------------------------------------------------

LOCAL_RANK: 1 - CUDA_VISIBLE_DEVICES: [0,1]
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]

  | Name                | Type          | Params | Mode 
--------------------------------------------------------------
0 | loss                | MAE           | 0      | train
1 | hist_cat_embeddings | ModuleList    | 0      | train
2 | futr_cat_embeddings | ModuleList    | 0      | train
3 | stat_cat_embeddings | ModuleList    | 0      | train
4 | padder_train        | ConstantPad1d | 0      | train
5 | scaler 

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Validation: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

`Trainer.fit` stopped: `max_steps=500` reached.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: |          | 0/? [00:00<?, ?it/s]

Number of rows in results after merge: 33796
Validation WMAE: 1533.28
Human-Readable Error (WAPE): 9.10%
🏃 View run NBEATS_Baseline at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/6/runs/45227a94584d405e9a261545628c3d5f
🧪 View experiment at: https://dagshub.com/ejoba22/walmart-sales-forecasting.mlflow/#/experiments/6
